In [1]:
import pandas as pd
import numpy as np
import os

In [3]:
# Define some paths
root = '.'
PATH_TO_CHARACTERIZATION = os.path.join(root, "..", "processed", "enamine_characterization")
PATH_TO_DOCKING_RESULTS = os.path.join(root, "..", "processed", "unidock_docking", 'docking_results')

# Load IDs
ids = open(os.path.join(PATH_TO_CHARACTERIZATION, "IDs.txt"), "r").readlines()
ids = np.array([i.strip() for i in ids])

# Load fingerprints
fps = np.load(os.path.join(PATH_TO_CHARACTERIZATION, "X.npz"))['X']

In [4]:
# For each pocket
for pocket in sorted(os.listdir(PATH_TO_DOCKING_RESULTS)):
        
        # Get scores
        data = pd.read_csv(os.path.join(PATH_TO_DOCKING_RESULTS, pocket, 'report.csv'))
        scores = data['score'].to_numpy()

        # Define percentiles
        percentiles = [0.1]

        # For each percentile
        for perc in percentiles:
                
                # Define score cutoff
                score_cutoff = np.percentile(scores, perc)

                # Define actives and inactives
                actives = data[data['score'] <= score_cutoff]['compound'].to_numpy()
                inactives = data[data['score'] > score_cutoff]['compound'].to_numpy()

                # Define indices actives (and, by definition, indices inactives
                ind_actives = np.where(np.isin(ids, actives))[0]

                # Boolean mask for actives
                mask_actives = np.zeros(len(ids), dtype=bool)
                mask_actives[ind_actives] = True

                # Slice fingerprints
                X_act = fps[mask_actives]
                X_inact = fps[~mask_actives]

                # Labels
                y_act = np.ones(len(X_act), dtype=int)
                y_inact = np.zeros(len(X_inact), dtype=int)

                # Combine and shuffle
                X_combined = np.vstack([X_act, X_inact])
                y_combined = np.hstack([y_act, y_inact])
                perm = np.random.permutation(len(y_combined))
                X = X_combined[perm]
                y = y_combined[perm]

        break

In [5]:
import lazyqsar as lq

You are not using the full version of lazy-qsar which has descriptors pipeline!
invalid syntax (qsar.py, line 11)


/home/acomajuncosa/miniconda3/envs/lazyqsar/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
!pip install torch

  Using cached torch-2.7.1-cp311-cp311-manylinux_2_28_x86_64.whl.metadata (29 kB)
  Using cached filelock-3.18.0-py3-none-any.whl.metadata (2.9 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.5-py3-none-any.whl.metadata (6.3 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2025.7.0-py3-none-any.whl.metadata (12 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.6.77-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_runtime_cu12-12.6.77-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cuda_cupti_cu12-12.6.80-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cudnn_cu12-9.5.1.17-py3-none-manylinux_2_28_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_cublas_cu12-12.6.4.1-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.5 kB)
  Using cached nvidia_cufft_cu12-11.3.0.4-py3-n

In [22]:
from lazyqsar.qsar import LazyBinaryQSAR

In [25]:
lq = LazyBinaryQSAR(descriptor_type='chemeleon', model_type='random_forest')

In [32]:
lq.fit(X[:1000], y[:1000])

Fitting inputs to feature descriptors using chemeleon
No fitting is necessary for Chemeleon descriptor
Transforming inputs to feature descriptors using chemeleon


Transforming CheMeleon descriptors in chunks of 1000:   0%|          | 0/1 [00:00<?, ?it/s]


AttributeError: 'numpy.ndarray' object has no attribute 'GetNumAtoms'